# Вариационный вывод для анализа финансовых данных
## Применение байесовских методов к данным Московской биржи

**Предмет:** Методы искусственного интеллекта  
**Тема:** 3.2 — Вариационный вывод (Трейдинг)

---
### Постановка задачи

Дано: временной ряд цен акций $\{S_t\}_{t=1}^{T}$ с Московской биржи.

Задача: оценить параметры распределения логарифмических доходностей:
$$r_t = \ln\frac{S_t}{S_{t-1}}$$

используя **вариационный вывод** (Variational Inference) вместо точного байесовского вычисления.


In [ ]:
%matplotlib inline
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.figsize": (12,5), "font.size":12})
print("Библиотеки загружены успешно")


## 1. Загрузка данных с Московской биржи (MOEX ISS API)

Используем открытый REST API Московской биржи. Запрос возвращает дневные свечи (OHLCV) для заданного тикера.

Логарифмические доходности вычисляются как:
$$r_t = \ln S_t - \ln S_{t-1}$$

Преимущество лог-доходностей: они аддитивны по времени и приближённо нормально распределены.


In [ ]:
TICKER = "SBER"  # Сбербанк

date_till = datetime.today().strftime("%Y-%m-%d")
date_from = (datetime.today() - timedelta(days=365*3)).strftime("%Y-%m-%d")
url = (f"https://iss.moex.com/iss/engines/stock/markets/shares/"
       f"securities/{TICKER}/candles.json"
       f"?from={date_from}&till={date_till}&interval=24&iss.meta=off")

resp = requests.get(url, timeout=15)
data = resp.json()
df = pd.DataFrame(data["candles"]["data"], columns=data["candles"]["columns"])
df["begin"] = pd.to_datetime(df["begin"])
df = df.set_index("begin").sort_index()

prices = df["close"].dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

print(f"Тикер: {TICKER}")
print(f"Период: {prices.index[0].date()} — {prices.index[-1].date()}")
print(f"Количество торговых дней: {len(log_returns)}")
print(f"Средняя дневная доходность: {log_returns.mean()*100:+.4f}%")
print(f"Дневная волатильность (std): {log_returns.std()*100:.4f}%")
print(f"Годовая волатильность: {log_returns.std()*np.sqrt(252)*100:.1f}%")


## 2. Анализ исходных данных

Перед построением моделей проведём разведочный анализ данных (EDA).

Гистограмма доходностей позволяет визуально оценить форму распределения.  
QQ-plot проверяет отклонение от нормального распределения (тяжёлые хвосты — типичное явление для финансовых данных).


In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Анализ данных: {TICKER}  |  {log_returns.index[0].date()} — {log_returns.index[-1].date()}"
             f"  |  Годовая волатильность: {log_returns.std()*np.sqrt(252)*100:.1f}%", fontsize=13, fontweight="bold")

# Цена
ax = axes[0,0]
ax.plot(prices.index, prices.values, color="#2196F3", linewidth=1.2)
ax.set_title("Цена закрытия (руб.)", fontweight="bold")
ax.set_ylabel("Цена (руб.)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.annotate(f"Старт: {prices.iloc[0]:.0f} руб.", xy=(prices.index[0], prices.iloc[0]),
            xytext=(10,10), textcoords="offset points", fontsize=9, color="green")
ax.annotate(f"Конец: {prices.iloc[-1]:.0f} руб.", xy=(prices.index[-1], prices.iloc[-1]),
            xytext=(-70,10), textcoords="offset points", fontsize=9, color="red")

# Доходности
ax = axes[0,1]
colors = ["#4CAF50" if r>=0 else "#F44336" for r in log_returns.values]
ax.bar(log_returns.index, log_returns.values, color=colors, alpha=0.7, width=1)
ax.axhline(0, color="black", linestyle="--", alpha=0.5)
ax.axhline(log_returns.mean(), color="blue", linewidth=1.5, label=f"Среднее: {log_returns.mean()*100:+.4f}%")
ax.set_title("Дневные доходности (зелёный=рост, красный=падение)", fontweight="bold")
ax.legend(fontsize=9)

# Гистограмма
ax = axes[1,0]
ax.hist(log_returns.values, bins=80, density=True, color="#FF9800", alpha=0.7, edgecolor="white")
ax.axvline(log_returns.mean(), color="red", linewidth=2, label=f"μ = {log_returns.mean()*100:+.4f}%")
ax.axvline(log_returns.mean()+log_returns.std(), color="purple", linestyle="--", label=f"+1σ")
ax.axvline(log_returns.mean()-log_returns.std(), color="purple", linestyle="--", label=f"-1σ")
ax.set_title("Распределение доходностей — 'колокольчик'", fontweight="bold")
ax.set_xlabel("Log Return"); ax.set_ylabel("Плотность")
ax.legend(fontsize=9)

# QQ-plot
ax = axes[1,1]
sorted_r = np.sort(log_returns.values)
theor_q = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_r)))
ax.scatter(theor_q, sorted_r, alpha=0.3, s=5, color="#9C27B0")
ax.plot([-4,4], [-4*log_returns.std(), 4*log_returns.std()], "r--", alpha=0.8, label="Идеальная нормальность")
ax.set_title("QQ-plot: проверка нормальности\n(точки на линии = нормальное распределение)", fontweight="bold")
ax.set_xlabel("Теоретические квантили"); ax.set_ylabel("Реальные квантили")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## 3. Модель 1: Нормальное распределение доходностей

### Математическое описание

**Модель:**
$$r_t \sim \mathcal{N}(\mu, \sigma^2)$$

**Априорные распределения (prior):**
$$\mu \sim \mathcal{N}(0,\ 0.1^2)$$
$$\sigma \sim \text{HalfNormal}(0.1)$$

**Функция правдоподобия (likelihood):**
$$p(r_{1:T} | \mu, \sigma) = \prod_{t=1}^T \mathcal{N}(r_t | \mu, \sigma^2)$$

### Вариационный вывод (ADVI)

Вместо точного вычисления апостериорного распределения $p(\mu,\sigma|r_{1:T})$, аппроксимируем его семейством нормальных распределений $q(\mu,\sigma)$, минимизируя KL-дивергенцию:

$$q^*(\theta) = \arg\min_{q \in \mathcal{Q}} \text{KL}[q(\theta) \| p(\theta | r_{1:T})]$$

Эквивалентно максимизации ELBO:
$$\text{ELBO}(q) = \mathbb{E}_{q}[\log p(r_{1:T}, \theta)] - \mathbb{E}_{q}[\log q(\theta)]$$


In [ ]:
import pymc as pm

returns_data = log_returns.values.astype(np.float64)

with pm.Model() as model1:
    # Априорные распределения
    mu = pm.Normal("mu", mu=0, sigma=0.1)
    sigma = pm.HalfNormal("sigma", sigma=0.1)
    
    # Функция правдоподобия
    likelihood = pm.Normal("returns", mu=mu, sigma=sigma, observed=returns_data)
    
    # Вариационный вывод — ADVI
    print("Запуск ADVI (Автоматический Дифференцируемый Вариационный Вывод)...")
    approx1 = pm.fit(n=30000, method="advi", progressbar=True)
    trace1 = approx1.sample(5000)

print("\nМодель 1 обучена!")
mu_samples = trace1.posterior["mu"].values.flatten()
sigma_samples = trace1.posterior["sigma"].values.flatten()
print(f"μ = {mu_samples.mean()*100:+.4f}% ± {mu_samples.std()*100:.4f}%")
print(f"σ = {sigma_samples.mean()*100:.4f}% в день ≈ {sigma_samples.mean()*np.sqrt(252)*100:.1f}% годовых")


In [ ]:
from scipy.stats import norm

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Модель 1 — {TICKER}\nБайесовские оценки параметров (ADVI)", fontsize=13, fontweight="bold")

# Апостериорное распределение μ
ax = axes[0]
ax.hist(mu_samples, bins=60, density=True, color="#2196F3", alpha=0.7, edgecolor="white")
ax.axvline(mu_samples.mean(), color="red", linewidth=2, label=f"μ = {mu_samples.mean()*100:+.4f}%/день")
ax.axvspan(np.percentile(mu_samples,5), np.percentile(mu_samples,95),
           alpha=0.15, color="blue", label="90% доверительный интервал")
ax.set_title("μ — средняя дневная доходность\n(чем правее, тем лучше — акция растёт)", fontweight="bold")
ax.set_xlabel("Значение μ"); ax.set_ylabel("Плотность вероятности")
ax.legend(fontsize=9)

# Апостериорное распределение σ
ax = axes[1]
ax.hist(sigma_samples, bins=60, density=True, color="#FF9800", alpha=0.7, edgecolor="white")
ax.axvline(sigma_samples.mean(), color="red", linewidth=2,
           label=f"σ = {sigma_samples.mean()*100:.3f}%/день\n≈ {sigma_samples.mean()*np.sqrt(252)*100:.1f}% год.")
ax.axvspan(np.percentile(sigma_samples,5), np.percentile(sigma_samples,95),
           alpha=0.15, color="orange", label="90% доверительный интервал")
ax.set_title("σ — волатильность (риск)\nГодовая волатильность ~24% = умеренный риск", fontweight="bold")
ax.set_xlabel("Значение σ"); ax.set_ylabel("Плотность вероятности")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Плотность распределения предсказаний
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(f"Модель 1 — {TICKER}\nПлотность распределения предсказаний vs реальные данные",
             fontsize=13, fontweight="bold")

ax.hist(log_returns.values, bins=80, density=True, color="#BDBDBD", alpha=0.7,
        edgecolor="white", label="Реальные доходности")

x = np.linspace(log_returns.min(), log_returns.max(), 300)
for i in range(200):
    idx = np.random.randint(len(mu_samples))
    y = norm.pdf(x, mu_samples[idx], sigma_samples[idx])
    ax.plot(x, y, color="#2196F3", alpha=0.02)

y_mean = norm.pdf(x, mu_samples.mean(), sigma_samples.mean())
ax.plot(x, y_mean, color="red", linewidth=2.5,
        label=f"Средняя модель: N({mu_samples.mean()*100:+.4f}%, {sigma_samples.mean()*100:.3f}%)")

ax.set_xlabel("Log Return"); ax.set_ylabel("Плотность")
ax.set_title("Синие кривые = 200 вариантов модели (неопределённость параметров)\n"
             "Красная = средняя оценка модели", fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 4. Модель 2: Стохастическая волатильность

### Математическое описание

**Мотивация:** В Модели 1 волатильность $\sigma$ постоянна. Но реальные рынки показывают **кластеризацию волатильности** — в кризис она высокая, в спокойное время — низкая.

**Модель стохастической волатильности:**

Скрытый процесс (лог-волатильность как случайное блуждание):
$$h_t = h_{t-1} + \sigma_h \cdot \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0,1)$$

Наблюдаемые доходности:
$$r_t \sim \mathcal{N}\left(0,\ e^{h_t}\right)$$

**Априорные распределения:**
$$\sigma_h \sim \text{HalfNormal}(0.5)$$
$$h_0 \sim \mathcal{N}(-5, 2)$$

Мгновенная волатильность в момент $t$:
$$\sigma_t = e^{h_t/2}$$

### Преимущество над Моделью 1

Модель 2 позволяет восстановить **траекторию волатильности** $\{\sigma_t\}$ — инструмент для оценки риска в реальном времени.


In [ ]:
with pm.Model() as model2:
    sigma_h = pm.HalfNormal("sigma_h", sigma=0.5)
    h0 = pm.Normal("h0", mu=-5, sigma=2)
    
    # Случайное блуждание — скрытая лог-волатильность
    h = pm.GaussianRandomWalk("h", sigma=sigma_h,
                               init_dist=pm.Normal.dist(mu=h0, sigma=0.1),
                               steps=len(returns_data)-1)
    volatility = pm.math.exp(h / 2)
    obs = pm.Normal("returns", mu=0, sigma=volatility, observed=returns_data)
    
    print("Запуск ADVI для Модели 2 (стохастическая волатильность)...")
    approx2 = pm.fit(n=50000, method="advi", progressbar=True)
    trace2 = approx2.sample(3000)

print("\nМодель 2 обучена!")


In [ ]:
dates = log_returns.index
h_samples = trace2.posterior["h"].values
h_mean = h_samples.mean(axis=(0,1))
h_low  = np.percentile(h_samples, 5, axis=(0,1))
h_high = np.percentile(h_samples, 95, axis=(0,1))
vol_mean = np.exp(h_mean/2) * 100
vol_low  = np.exp(h_low/2) * 100
vol_high = np.exp(h_high/2) * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
fig.suptitle(f"Модель 2: Стохастическая волатильность — {TICKER}\n"
             "Волатильность НЕ постоянная — меняется каждый день",
             fontsize=13, fontweight="bold")

# Цена
ax = axes[0]
ax.plot(dates, prices.loc[dates].values, color="#2196F3", linewidth=1.2)
ax.fill_between(dates, prices.loc[dates].values, alpha=0.1, color="#2196F3")
ax.set_ylabel("Цена (руб.)"); ax.set_title("Цена акции", fontweight="bold")

# Доходности
ax = axes[1]
colors_r = ["#4CAF50" if r>=0 else "#F44336" for r in log_returns.values]
ax.bar(dates, log_returns.values, color=colors_r, alpha=0.7, width=1)
ax.axhline(0, color="gray", linestyle="--")
ax.set_ylabel("Доходность")
ax.set_title("Дневные доходности (зелёный=рост, красный=падение)", fontweight="bold")

# Волатильность
ax = axes[2]
ax.plot(dates, vol_mean, color="#F44336", linewidth=2, label="Оценённая волатильность")
ax.fill_between(dates, vol_low, vol_high, color="#F44336", alpha=0.2, label="90% доверительный интервал")
max_idx = np.argmax(vol_mean)
ax.annotate(f"Пик риска\n{dates[max_idx].date()}",
            xy=(dates[max_idx], vol_mean[max_idx]),
            xytext=(30,10), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
ax.set_ylabel("Волатильность (%)"); ax.set_xlabel("Дата")
ax.set_title("Оценённая волатильность — ПИКИ = периоды наибольшего риска", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. Выводы

### Сравнение моделей

| Характеристика | Модель 1 (Normal) | Модель 2 (SV) |
|---|---|---|
| Предположение о волатильности | Постоянная: $\sigma = const$ | Переменная: $\sigma_t = e^{h_t/2}$ |
| Число параметров | 2 $(\mu, \sigma)$ | $T+2$ $(h_1,...,h_T, \sigma_h, h_0)$ |
| Гибкость | Низкая | Высокая |
| Интерпретируемость | Простая | Богатая |

### Результаты для Сбербанка (2023–2025)

- **Средняя дневная доходность μ:** близка к нулю, 90% интервал включает 0 → нет уверенного тренда
- **Годовая волатильность σ:** ~24% → умеренно рискованный актив (для сравнения: S&P500 ~15-20%)
- **Стохастическая волатильность:** модель фиксирует периоды повышенного риска, соответствующие рыночным событиям

### Заключение

Вариационный вывод (ADVI) позволяет эффективно оценивать параметры байесовских моделей на реальных финансовых данных без точного вычисления апостериорного распределения. Модель стохастической волатильности лучше описывает реальную динамику рынка благодаря учёту изменчивости риска во времени.
